##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [48]:
# Import libraries
import os
import cv2
import random
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model


In [26]:
# Setting the seed value to ensure reproducibility of results
seed_constant = 23
np.random.seed(seed_constant)
random.seed(seed_constant)
tf.random.set_seed(seed_constant)

In [3]:
# Check the classes of the dataset
DATASET_DIR = "UCF11/UCF11_updated_mpg"
print(os.listdir(DATASET_DIR))

['basketball', 'biking', 'diving', 'golf_swing', 'horse_riding', 'soccer_juggling', 'swing', 'tennis_swing', 'trampoline_jumping', 'volleyball_spiking', 'walking']


In [5]:
# Choose 3 classes from the dataset
CLASSES_LIST = ["basketball", "biking", "swing"]

In [40]:
# Preprocess the dataset and extract frames
image_height, image_width = 64, 64
max_images_per_class = 8000
model_output_size = len(CLASSES_LIST)

def frames_extraction(video_path):
    # Empty List declared to store video frames
    frames_list = []
    
    # Reading the Video File Using the VideoCapture
    video_reader = cv2.VideoCapture(video_path)

    # Iterating through Video Frames
    while True:

        # Reading a frame from the video file 
        success, frame = video_reader.read() 

        if not success:
            break

        # Resize the Frame to fixed Dimensions
        resized_frame = cv2.resize(frame, (image_height, image_width))
        
        # Normalize the resized frame 
        normalized_frame = resized_frame / 255
        
        # Appending the normalized frame into the frames list
        frames_list.append(normalized_frame)
    
    # Closing the VideoCapture object and releasing all resources. 
    video_reader.release()

    # returning the frames list 
    return frames_list

In [ ]:
# Create the dataset for the specified classes 
def create_dataset():
    temp_features = [] 
    features = []
    labels = []
    
    # Iterating through all the classes mentioned in the classes list
    for class_index, class_name in enumerate(CLASSES_LIST):
        print(f'Extracting Data of Class: {class_name}')
        
        class_path = os.path.join(DATASET_DIR, class_name)

        # use os.walk instead of os.listdir
        for root, dirs, files in os.walk(class_path):

            for file_name in files:

                # only process video files
                if file_name.lower().endswith((".mpg", ".avi", ".mp4", ".mov")):

                    # Construct the complete video path
                    video_file_path = os.path.join(root, file_name)

                    # Calling the frame_extraction method
                    frames = frames_extraction(video_file_path)

                    # Appending the frames to a temporary list.
                    temp_features.extend(frames)
        
        print(f"{class_name}: {len(temp_features)} frames")

        # avoid crash if not enough frames
        sample_size = min(len(temp_features), max_images_per_class)

        # Adding randomly selected frames to the features list
        features.extend(random.sample(temp_features, sample_size))

        # Adding labels
        labels.extend([class_index] * sample_size)
        
        # Emptying temp_features for next class
        temp_features.clear()

    # Converting to numpy arrays
    features = np.asarray(features)
    labels = np.array(labels)  

    return features, labels


features, labels = create_dataset()
one_hot_encoded_labels = to_categorical(labels)

Extracting Data of Class: basketball
basketball: 19230 frames
Extracting Data of Class: biking
biking: 32863 frames
Extracting Data of Class: swing
swing: 27672 frames


In [42]:
# Splitting the dataset into training and testing sets
features_train, features_test, labels_train, labels_test = train_test_split(features, one_hot_encoded_labels, test_size = 0.2, shuffle = True, random_state = seed_constant)

In [43]:
# Create the model
def create_model():

    model = Sequential()

    # Defining The Model Architecture
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu', input_shape = (image_height, image_width, 3)))
    model.add(Conv2D(filters = 64, kernel_size = (3, 3), activation = 'relu'))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size = (2, 2)))
    model.add(GlobalAveragePooling2D())
    model.add(Dense(256, activation = 'relu'))
    model.add(BatchNormalization())
    model.add(Dense(model_output_size, activation = 'softmax'))

    # Printing the models summary
    model.summary()

    return model


# Calling the create_model method
model = create_model()

print("Model Created Successfully!")

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 62, 62, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 60, 60, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,411 (224.26 KB)

 Trainable params: 56,771 (221.76 KB)

 Non-trainable params: 640 (2.50 KB)

Model Created Successfully!


In [45]:
# Defining the Early Stopping Callback
early_stopping_callback = EarlyStopping(monitor = 'val_loss', patience = 15, 
                                        mode = 'min', restore_best_weights = True)

# Adding loss, optimizer and metrics values to the model.
model.compile(loss = 'categorical_crossentropy', optimizer = 'Adam', metrics = ["accuracy"])

# Start Training
model_training_history = model.fit(x = features_train, y = labels_train, epochs = 30, batch_size = 4 ,
                                    shuffle = True, validation_split = 0.2, callbacks = [early_stopping_callback])

Epoch 1/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 103s 26ms/step - accuracy: 0.6874 - loss: 0.7318 - val_accuracy: 0.6232 - val_loss: 0.8953
Epoch 2/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 91s 24ms/step - accuracy: 0.7964 - loss: 0.5183 - val_accuracy: 0.8956 - val_loss: 0.3367
Epoch 3/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 95s 25ms/step - accuracy: 0.8406 - loss: 0.4191 - val_accuracy: 0.8044 - val_loss: 0.6098
Epoch 4/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 143s 37ms/step - accuracy: 0.8649 - loss: 0.3621 - val_accuracy: 0.9510 - val_loss: 0.1499
Epoch 5/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 90s 24ms/step - accuracy: 0.8844 - loss: 0.3181 - val_accuracy: 0.8802 - val_loss: 0.3503
Epoch 6/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 99s 26ms/step - accuracy: 0.9016 - loss: 0.2806 - val_accuracy: 0.8479 - val_loss: 0.5615
Epoch 7/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 97s 25ms/step - accuracy: 0.9076 - loss: 0.2647 - val_accuracy: 0.9167 - val_loss: 0.2501
Epoch 8/30
3840/3840 ━━━━━━━━━━━━━━━━━━━━ 134s 35ms/step - accuracy: 0.914

In [46]:
loss, accuracy = model.evaluate(features_test, labels_test)
print("Test Accuracy:", accuracy)

150/150 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.9802 - loss: 0.0634
Test Accuracy: 0.9802083373069763


In [47]:
# Save the model
student_name = "Haya_Aldossary"
save_path = f"{student_name}_ucf11_model.h5"

model.save(save_path)
print(f"Model saved as {save_path}")

Model saved as Haya_Aldossary_ucf11_model.h5


In [68]:
# Test the model on a 3 youtube videos
model = load_model("Haya_Aldossary_ucf11_model.h5")

def predict_video(video_path):
    frames = frames_extraction(video_path)

    if len(frames) == 0:
        print("No frames extracted from video.")
        return

    predictions = []

    for frame in frames:
        frame = np.expand_dims(frame, axis=0)
        pred = model.predict(frame, verbose=0)
        predictions.append(pred[0])

    avg_prediction = np.mean(predictions, axis=0)

    predicted_class_index = np.argmax(avg_prediction)
    predicted_class_name = CLASSES_LIST[predicted_class_index]
    confidence = avg_prediction[predicted_class_index]

    print("Video:", video_path)
    print("Predicted Action:", predicted_class_name)
    print("Confidence:", round(confidence * 100, 2), "%")

test_videos = [
    r"C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\basketball.mp4",
    r"C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\biking.mp4",
    r"C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\swing.mp4"
]

for video_file in test_videos:
    predict_video(video_file)
    print("-" * 30)

Video: C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\basketball.mp4
Predicted Action: basketball
Confidence: 66.52 %
------------------------------
Video: C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\biking.mp4
Predicted Action: biking
Confidence: 88.04 %
------------------------------
Video: C:\Users\hayal\OneDrive\Documents\GitHub\arti560-computer-vision-labs\lab07-action-recognition\youtube_video_test\swing.mp4
Predicted Action: swing
Confidence: 91.65 %
------------------------------
